# Phase 05 · Bài 02 — Bag of Words và TF-IDF

**Mục tiêu:** hiểu vocabulary xác định không gian đặc trưng, tự tính BoW/TF-IDF, chuẩn hóa vector và dùng cosine để xếp hạng tài liệu.

## 1. Từ chuỗi sang ma trận

Giả sử corpus có N tài liệu và vocabulary gồm V token. Ta lập ma trận X có kích thước N × V; phần tử X[i,j] là số lần token j xuất hiện trong tài liệu i. Mỗi hàng là một tài liệu, mỗi cột là một đặc trưng từ vựng. Mọi tài liệu trong corpus sau đó có cùng số chiều dù độ dài câu khác nhau.

Đây là Bag of Words vì phép đếm bỏ qua thứ tự. “dog bites man” và “man bites dog” có cùng vector unigram. BoW vẫn mạnh trên các tác vụ mà sự hiện diện/tần suất từ đã chứa đủ tín hiệu, đồng thời dễ giải thích: biết cột nào có giá trị lớn là biết từ nào xuất hiện.

Vocabulary được xây từ corpus huấn luyện và giữ cố định khi suy luận. Token ngoài vocabulary thường bị bỏ hoặc ánh xạ sang một mục UNK nếu pipeline có định nghĩa mục đó. Nếu tạo vocabulary hoặc IDF từ dữ liệu test thì quy trình đánh giá không còn tách biệt đúng với dữ liệu huấn luyện.

## 2. TF và IDF làm hai việc khác nhau

Term Frequency (TF) đo mức độ một từ xuất hiện trong một tài liệu. Code bài dùng TF chuẩn hóa theo độ dài: count(w,d) / số token trong d. Nếu chỉ dùng count, tài liệu dài có thể nhận vector lớn hơn chỉ vì có nhiều từ.

Document Frequency (df) là số tài liệu có chứa từ đó, không phải tổng số lần xuất hiện. Inverse Document Frequency (IDF) giảm trọng số từ xuất hiện gần như ở mọi nơi và tăng trọng số từ ít phổ biến hơn. Trong implementation bài học:

idf(w) = log((N + 1)/(df(w) + 1)) + 1

Smoothing bằng cộng 1 tránh chia cho 0; cộng 1 bên ngoài khiến từ có mặt trong mọi tài liệu vẫn có trọng số IDF bằng 1 thay vì 0. Một số thư viện dùng biến thể công thức khác, nên con số TF-IDF có thể khác đôi chút dù xu hướng giống nhau.

TF-IDF(w,d) = TF(w,d) × IDF(w)

Từ xuất hiện nhiều trong một tài liệu nhưng hiếm toàn corpus thường trở thành feature nổi bật. Từ chung như “the” bị giảm ảnh hưởng. Vector thường thưa vì một tài liệu chỉ dùng một phần nhỏ trong toàn vocabulary.

## 3. L2 normalization và cosine

Chuẩn hóa L2 chia từng thành phần cho norm của hàng: norm(x) = sqrt(sum(x_j²)). Vector sau đó có độ dài 1, trừ vector toàn 0. Cosine similarity là (a·b)/(||a|| ||b||); với hai hàng đã chuẩn hóa, phép tính rút thành dot product. Cosine gần 1 nghĩa là hướng vector gần nhau; gần 0 nghĩa là ít feature chung theo biểu diễn này.

Code mẫu tạo ba câu, hiển thị vocabulary, các hàng TF-IDF và ma trận cosine. Hai câu cùng “cat/sat/on/mat” đạt cosine 0.80; câu ít từ chung hơn với câu đầu đạt 0.46. Đây là tương đồng trong không gian từ, không phải phán đoán hai câu đồng nghĩa.

## 4. Giới hạn ngữ nghĩa và cách mở rộng

Unigram TF-IDF không giữ trật tự nên dễ bỏ lỡ phủ định: “good” và “not good” cùng chứa feature “good”. N-gram thêm các cụm liên tiếp như “not_good” và có thể nắm một số mẫu cục bộ, nhưng làm vocabulary lớn hơn, thưa hơn và vẫn không hiểu cú pháp tổng quát.

TF-IDF không tự biết từ đồng nghĩa (“refund” và “reimbursement”) và không đại diện tốt cho từ lạ chưa thấy. Bù lại, nó nhanh, rẻ, dễ kiểm tra và thường là baseline nên thử trước cho phân loại hẹp hoặc tìm kiếm từ khóa. Notebook thực hành fit vocabulary/IDF trên các ticket mẫu, biến query bằng đúng thống kê corpus đó rồi xếp hạng cosine; ticket về refund bị trễ đứng đầu với score 0.738.

## A. Chạy code mẫu của bài

Cell này thực thi code/main.py nguyên bản.

In [1]:
from pathlib import Path
import subprocess
import sys

PHASE = "phases/05-nlp-foundations-to-advanced"
LESSON = "02-bag-of-words-tfidf"
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / PHASE / "README.md").is_file():
        REPO = candidate
        break
else:
    raise FileNotFoundError("Mở Jupyter từ thư mục repo hoặc thư mục con của repo.")
script = REPO / PHASE / LESSON / "code" / "main.py"
result = subprocess.run([sys.executable, str(script)], cwd=REPO, text=True, capture_output=True)
print("Bài:", "Bag of Words and TF-IDF")
print("Lệnh:", script.relative_to(REPO))
print("Python:", sys.version.split()[0])
print("Exit code:", result.returncode)
print("\n--- stdout ---")
print(result.stdout.rstrip())
if result.stderr:
    print("\n--- stderr ---")
    print(result.stderr.rstrip())
if result.returncode != 0:
    raise RuntimeError(f"Code mẫu của {LESSON} thoát với mã {result.returncode}")


Bài: Bag of Words and TF-IDF
Lệnh: phases\05-nlp-foundations-to-advanced\02-bag-of-words-tfidf\code\main.py
Python: 3.12.10
Exit code: 0

--- stdout ---
vocab: ['the', 'cat', 'sat', 'on', 'mat', 'dog', 'ran', 'across', 'room']

d0: The cat sat on the mat.
    tfidf: ['0.61', '0.39', '0.39', '0.39', '0.39', '0.00', '0.00', '0.00', '0.00']

d1: The dog sat on the mat.
    tfidf: ['0.58', '0.00', '0.37', '0.37', '0.37', '0.49', '0.00', '0.00', '0.00']

d2: The cat ran across the room.
    tfidf: ['0.53', '0.34', '0.00', '0.00', '0.00', '0.00', '0.45', '0.45', '0.45']

cosine similarity matrix:
  d0: ['1.00', '0.80', '0.46']
  d1: ['0.80', '1.00', '0.31']
  d2: ['0.46', '0.31', '1.00']


## B. Thực hành riêng — xếp hạng ticket

Mỗi hàng của ma trận là một tài liệu, mỗi cột là một từ. Ta in vocabulary, df và idf, rồi biến query thành vector trong cùng không gian và dùng IDF của corpus trước khi xếp hạng bằng cosine.

In [2]:
import math
import re

TOKEN_RE = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?|[0-9]+")
def tokenize(text):
    return [token.lower() for token in TOKEN_RE.findall(text)]

tickets = [
    "refund was approved within five business days",
    "refund delayed because the bank transfer is pending",
    "tracking shows the parcel is out for delivery",
    "the damaged parcel needs a replacement",
]
docs = [tokenize(text) for text in tickets]

def build_vocab(documents):
    vocab = {}
    for doc in documents:
        for token in doc:
            if token not in vocab:
                vocab[token] = len(vocab)
    return vocab

def bow_matrix(documents, vocab):
    rows = [[0] * len(vocab) for _ in documents]
    for row_index, doc in enumerate(documents):
        for token in doc:
            if token in vocab:
                rows[row_index][vocab[token]] += 1
    return rows

def corpus_tfidf(counts):
    n_docs, n_terms = len(counts), len(counts[0])
    df = [sum(row[j] > 0 for row in counts) for j in range(n_terms)]
    idf = [math.log((n_docs + 1) / (df_j + 1)) + 1 for df_j in df]
    vectors = []
    for row in counts:
        length = sum(row)
        values = [(count / length) * idf[j] if length else 0.0
                  for j, count in enumerate(row)]
        norm = math.sqrt(sum(value * value for value in values))
        vectors.append([value / norm if norm else 0.0 for value in values])
    return vectors, df, idf

def cosine(left, right):
    return sum(x * y for x, y in zip(left, right))

vocab = build_vocab(docs)
counts = bow_matrix(docs, vocab)
vectors, df, idf = corpus_tfidf(counts)
query = "refund delayed bank transfer"
query_tokens = tokenize(query)
query_counts = bow_matrix([query_tokens], vocab)[0]
query_tf = [count / len(query_tokens) for count in query_counts]
query_values = [tf * idf[j] for j, tf in enumerate(query_tf)]
query_norm = math.sqrt(sum(value * value for value in query_values))
query_vector = [value / query_norm if query_norm else 0.0 for value in query_values]
ranking = sorted(((cosine(query_vector, vector), i)
                  for i, vector in enumerate(vectors)), reverse=True)

print("vocabulary size:", len(vocab))
print("vocabulary (first 15):", list(vocab)[:15])
print("count row for first ticket:", counts[0])
print("document frequency for refund:", df[vocab["refund"]])
print("idf for refund:", round(idf[vocab["refund"]], 3))
print("\nquery:", query)
for score, index in ranking:
    print(f"{score:.3f}  {tickets[index]}")

vocabulary size: 24
vocabulary (first 15): ['refund', 'was', 'approved', 'within', 'five', 'business', 'days', 'delayed', 'because', 'the', 'bank', 'transfer', 'is', 'pending', 'tracking']
count row for first ticket: [1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
document frequency for refund: 2
idf for refund: 1.511

query: refund delayed bank transfer
0.738  refund delayed because the bank transfer is pending
0.127  refund was approved within five business days
0.000  the damaged parcel needs a replacement
0.000  tracking shows the parcel is out for delivery


## C. Thí nghiệm — unigram và bigram

Chỉ đếm unigram khiến good và not good cùng có feature good. Bigram not_good giữ được tín hiệu phủ định. Thêm ticket hoặc đổi query rồi quan sát ranking.

In [3]:
def add_bigrams(tokens):
    return tokens + [f"{left}_{right}" for left, right in zip(tokens, tokens[1:])]

reviews = ["the movie was good", "the movie was not good"]
for review in reviews:
    unigrams = tokenize(review)
    print(review)
    print("  unigram:", unigrams)
    print("  unigram + bigram:", add_bigrams(unigrams))

print("\nBigram bổ sung tín hiệu cụm từ, nhưng không tự hiểu mọi cấu trúc câu.")

the movie was good
  unigram: ['the', 'movie', 'was', 'good']
  unigram + bigram: ['the', 'movie', 'was', 'good', 'the_movie', 'movie_was', 'was_good']
the movie was not good
  unigram: ['the', 'movie', 'was', 'not', 'good']
  unigram + bigram: ['the', 'movie', 'was', 'not', 'good', 'the_movie', 'movie_was', 'was_not', 'not_good']

Bigram bổ sung tín hiệu cụm từ, nhưng không tự hiểu mọi cấu trúc câu.


## D. Tự kiểm tra

1. df(t) khác tf(t,d) thế nào?
2. Tại sao cần L2-normalize trước khi dùng cosine?
3. Vì sao không nên xóa not theo stopword list khi phân loại sentiment?
4. Thêm một tài liệu không có từ nào thuộc query. Cosine của nó nên ra sao?